# Identificação do projeto

**Título do projeto:** Análise Global de Culinárias, Complexidade Culinária e Perfis Nutricionais.

**Integrantes do grupo:** Enzo Leitão, Giovani Simões, Luan Lucho e Sarah Maneira

**Turma:** Sistemas de Informação - 6⁰ período

**Descrição breve do problema ou contexto:** análise das receitas publicadas no Food.com entre 1999 e 2018 para comparar culinárias e restrições alimentares quanto a tempo de preparo, complexidade e perfil nutricional.

Preencha este notebook ao longo dos dois RAs. Registre decisões e interpretações em poucas frases e acrescente as células de código necessárias em cada etapa.

---

# RA1 — Data Ingestion, Cleaning and Initial Analysis

Nesta etapa, carregue, prepare e explore os dados do projeto.

## 1. Contexto e objetivo

Apresente brevemente o problema investigado e o que o grupo pretende descobrir com os dados.

**Resposta do grupo:**

Plataformas de receitas como o Food.com reúnem milhares de receitas enviadas pelos próprios usuários. Como o cadastro é livre, os dados chegam com problemas: tempos de preparo irreais, informações nutricionais guardadas como texto e culinária ou tipo de dieta registrados apenas em tags livres.

Queremos entender como diferentes culinárias e restrições alimentares se diferenciam em tempo de preparo, complexidade (número de etapas e de ingredientes) e perfil nutricional nas receitas publicadas entre 1999 e 2018. 

Nesta etapa, o objetivo é obter os dados de forma reproduzível, diagnosticar os problemas de qualidade e gerar uma base limpa e documentada para as próximas etapas.

## 2. Fonte e ingestão dos dados

- Identifique a fonte dos dados, incluindo link ou referência de acesso.
- Descreva brevemente o dataset e o que seus registros representam.
- Explique como os dados foram carregados e implemente a ingestão.

**Resposta do grupo:**

**Fonte:** Food.com Recipes and Interactions, arquivo `RAW_recipes.csv`, publicado no Kaggle por Shuyang Li (versão 2), a partir do trabalho de Majumder et al. (2019). Link: https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions. Licença informada no Kaggle: "Data files © Original Authors".

**O que os registros representam:** cada linha é uma receita publicada por um usuário no Food.com. A base tem 231.637 receitas e 12 colunas: nome, tempo de preparo, autor, data de publicação, tags, informação nutricional, passos, descrição e ingredientes.

**Como os dados são carregados:** o notebook baixa o dataset com a biblioteca `kagglehub`, direto do Kaggle. O arquivo é lido com `pd.read_csv`, e um `assert` confere se a base tem 231.637 linhas e 12 colunas.

In [ ]:
import os
os.environ["DISABLE_COLAB_CACHE"] = "true"

import ast
import numpy as np
import pandas as pd
import kagglehub

In [ ]:
import sys, pyarrow
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("pyarrow:", pyarrow.__version__)
print("kagglehub:", kagglehub.__version__)

Python: 3.13.16
pandas: 2.2.3
numpy: 2.1.3
pyarrow: 23.0.1
kagglehub: 1.0.2


In [ ]:
pasta = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")
DADOS = os.path.join(pasta, "RAW_recipes.csv")

try:
    df_original = pd.read_csv(DADOS)
except Exception as erro:
    raise RuntimeError("Não foi possível carregar a base.") from erro

assert df_original.shape == (231637, 12), "A base carregada não é a esperada (231.637 × 12)"

print("✓ Dados carregados com sucesso!")
print(f"Fonte: {DADOS}")
print(f"Tamanho do arquivo: {round(os.path.getsize(DADOS) / 1024**2)} MB")
print(f"Shape: {df_original.shape[0]} linhas × {df_original.shape[1]} colunas")
display(df_original.head())

100%|██████████| 267M/267M [00:02<00:00, 113MB/s]

Extracting files...


✓ Dados carregados com sucesso!
Fonte: /root/.cache/kagglehub/datasets/shuyangli94/food-com-recipes-and-user-interactions/versions/2/RAW_recipes.csv
Tamanho do arquivo: 281 MB
Shape: 231637 linhas × 12 colunas


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


## 3. Exploração inicial dos dados

Investigue a quantidade de registros e colunas, os tipos de dados, os valores ausentes e outras características relevantes da base.

In [ ]:
df_original.info() # tipos dos dados

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231637 entries, 0 to 231636
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   name            231636 non-null  object
 1   id              231637 non-null  int64 
 2   minutes         231637 non-null  int64 
 3   contributor_id  231637 non-null  int64 
 4   submitted       231637 non-null  object
 5   tags            231637 non-null  object
 6   nutrition       231637 non-null  object
 7   n_steps         231637 non-null  int64 
 8   steps           231637 non-null  object
 9   description     226658 non-null  object
 10  ingredients     231637 non-null  object
 11  n_ingredients   231637 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 21.2+ MB


In [ ]:
display(df_original.isna().sum()) # ausencias

,0
name,1
id,0
minutes,0
contributor_id,0
submitted,0
tags,0
nutrition,0
n_steps,0
steps,0
description,4979


In [ ]:
# Q6 - id único
print("id é único?", df_original["id"].is_unique)
print("ids repetidos:", df_original["id"].duplicated().sum())

id é único? True
ids repetidos: 0


In [ ]:
# Q7 - receitas repetidas (mesmo nome e mesmo autor)
duplicadas = df_original.duplicated(subset=["name", "contributor_id"]).sum()
print("Duplicadas por nome + autor:", duplicadas)

df_original[df_original.duplicated(subset=["name", "contributor_id"], keep=False)].sort_values("name")

Duplicadas por nome + autor: 5


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
22013,berry orange smoothies,121303,3,152995,2005-05-09,"['15-minutes-or-less', 'time-to-make', 'course...","[119.9, 1.0, 78.0, 0.0, 5.0, 1.0, 9.0]",2,['in a blender container combine orange juice ...,"""keep the ingredients for this recipe on hand ...","['orange juice', 'banana', 'berries', 'low-fat...",4
22096,berry orange smoothies,121254,3,152995,2005-05-09,"['15-minutes-or-less', 'time-to-make', 'course...","[119.9, 1.0, 78.0, 0.0, 5.0, 1.0, 9.0]",2,"['in blender , combine orange juice , mixed be...","""keep the ingredients for this recipe on hand ...","['banana', 'orange juice', 'berries', 'low-fat...",4
37728,cauliflower cheese pot pies,328195,45,232669,2008-09-30,"['60-minutes-or-less', 'time-to-make', 'course...","[860.7, 97.0, 24.0, 20.0, 40.0, 128.0, 18.0]",4,"['het the oven to 360f', 'mix the creme fraich...",these looked so appealing that i thought i wou...,"['creme fraiche', 'cheddar cheese', 'spring on...",10
37897,cauliflower cheese pot pies,421192,45,232669,2010-04-21,"['weeknight', '60-minutes-or-less', 'time-to-m...","[704.7, 80.0, 23.0, 18.0, 38.0, 118.0, 14.0]",6,"['preheat the oven to 356f', 'mix the creme fr...",last night i had a craving for cauliflower -- ...,"['creme fraiche', 'cheddar cheese', 'sharp che...",12
166850,pumpkin cheesecake cups,74161,25,4439,2003-10-28,"['30-minutes-or-less', 'time-to-make', 'course...","[207.0, 15.0, 51.0, 8.0, 16.0, 28.0, 7.0]",10,"['preheat oven to 350 degrees fahrenheit', 'pl...","i found this on another web site, and immediat...","['gingersnap cookies', 'part-skim ricotta chee...",8
166851,pumpkin cheesecake cups,99654,45,4439,2004-09-13,"['60-minutes-or-less', 'time-to-make', 'course...","[180.4, 14.0, 56.0, 8.0, 10.0, 25.0, 6.0]",12,"['preheat oven to 350 degrees', 'line 24 muffi...",tweeked eagle brand maple cheesecake recipe us...,"['low-fat cream cheese', 'condensed milk', 'pu...",9
206555,sweet potato pecan pie,374,65,1545,1999-08-30,"['weeknight', 'time-to-make', 'course', 'main-...","[590.0, 46.0, 183.0, 16.0, 18.0, 51.0, 24.0]",16,"['preheat oven to 425 degrees fahrenheit', 'wi...",sweet potato pecan pie,"['yams', 'butter', 'condensed milk', 'cinnamon...",14
206689,sweet potato pecan pie,378,75,1545,1999-10-04,"['weeknight', 'time-to-make', 'course', 'main-...","[323.0, 30.0, 69.0, 7.0, 7.0, 34.0, 11.0]",14,"['prepare pie shell , set aside', 'make sweet ...",sweet potato-pecan pie,"['pie crust', 'egg', 'fresh sweet potatoes', '...",14
213056,three bean casserole,82550,45,49304,2004-01-30,"['60-minutes-or-less', 'time-to-make', 'course...","[320.4, 15.0, 12.0, 27.0, 34.0, 26.0, 14.0]",9,['spray 9-inch glass baking dish with cooking ...,a healthy bean cassrole. the recipe comes from...,"['kidney beans', 'navy beans', 'black beans', ...",14
213255,three bean casserole,53661,50,49304,2003-02-09,"['60-minutes-or-less', 'time-to-make', 'course...","[457.8, 11.0, 65.0, 40.0, 49.0, 7.0, 25.0]",10,"['put lima , black and kidney beans in a colan...","this is a low fat, high fiber, high in flavor ...","['lima beans', 'black beans', 'dark red kidney...",10


In [ ]:
# Q8: tempo de preparo, procurando zeros e valores absurdos
display(df_original["minutes"].describe())
print("Receitas com 0 minutos:", (df_original["minutes"] == 0).sum())

,minutes
count,2.316370e+05
mean,9.398546e+03
std,4.461963e+06
min,0.000000e+00
25%,2.000000e+01
50%,4.000000e+01
75%,6.500000e+01
max,2.147484e+09


Receitas com 0 minutos: 1094


In [ ]:
# Q9: tempos muito altos (mais de 1 dia de preparo)
print("Receitas com mais de 1 dia (1440 min):", (df_original["minutes"] > 1440).sum())
display(df_original.nlargest(5, "minutes")[["name", "minutes"]])

Receitas com mais de 1 dia (1440 min): 2000


,name,minutes
144074,no bake granola balls,2147483647
109624,how to preserve a husband,1051200
106563,homemade fruit liquers,288000
38116,celtic druid s honey mead meade metheglin,259260
106963,homemade vanilla,259205


In [ ]:
# Q10: n_steps e n_ingredients dentro do domínio do dicionário
print("n_steps = 0:", (df_original["n_steps"] == 0).sum())
print("n_ingredients < 1:", (df_original["n_ingredients"] < 1).sum())
display(df_original[["n_steps", "n_ingredients"]].describe())

n_steps = 0: 1
n_ingredients < 1: 0


,n_steps,n_ingredients
count,231637.000000,231637.000000
mean,9.765499,9.051153
std,5.995128,3.734796
min,0.000000,1.000000
25%,6.000000,6.000000
50%,9.000000,9.000000
75%,12.000000,11.000000
max,145.000000,43.000000


In [ ]:
# Q11: datas válidas e dentro de 1999–2018
datas = pd.to_datetime(df_original["submitted"], errors="coerce")
print("Datas inválidas:", datas.isna().sum())
print("Menor data:", datas.min())
print("Maior data:", datas.max())

Datas inválidas: 0
Menor data: 1999-08-06 00:00:00
Maior data: 2018-12-04 00:00:00


In [ ]:
# Transforma o texto em lista
import ast
steps_lista = df_original["steps"].apply(ast.literal_eval)
ingredients_lista = df_original["ingredients"].apply(ast.literal_eval)
nutrition_lista = df_original["nutrition"].apply(ast.literal_eval)
tags_lista = df_original["tags"].apply(ast.literal_eval)
print("✓ Listas prontas.")

✓ Listas prontas.


In [ ]:
# Q13: n_steps e n_ingredients batem com o tamanho das listas?
print("n_steps não bate:", (steps_lista.apply(len) != df_original["n_steps"]).sum())
print("n_ingredients não bate:", (ingredients_lista.apply(len) != df_original["n_ingredients"]).sum())

n_steps não bate: 0
n_ingredients não bate: 0


In [ ]:
# Q14: nutrition tem sempre 7 valores? As calorias fazem sentido?
print("Receitas sem 7 valores:", (nutrition_lista.apply(len) != 7).sum())
calorias = nutrition_lista.str[0]
print("Calorias negativas:", (calorias < 0).sum())
display(calorias.describe())

Receitas sem 7 valores: 0
Calorias negativas: 0


,nutrition
count,231637.000000
mean,473.942425
std,1189.711374
min,0.000000
25%,174.400000
50%,313.400000
75%,519.700000
max,434360.200000


In [ ]:
# Q15: alguma receita ficou sem tag?
print("Receitas sem nenhuma tag:", (tags_lista.apply(len) == 0).sum())

Receitas sem nenhuma tag: 0


**Resposta do grupo:**

- A base tem 231.637 linhas e 12 colunas. Cada linha é uma receita, e o `id` é único (Q6).
- **Tipos:** `submitted` está como texto, e `tags`, `nutrition`, `steps` e `ingredients` estão como texto que representa lista (Q4 e Q5). Essas colunas precisam ser convertidas.
- **Ausências:** 1 receita sem nome (Q1) e 4.979 sem descrição (2,15% de 231.637), que é um campo opcional (Q2). As demais colunas estão completas (Q3).
- **Duplicidades:** 5 receitas cadastradas duas vezes pelo mesmo autor, com o mesmo nome e `id` diferente (Q7).
- **Tempo de preparo (`minutes`):** 1.094 receitas com 0 minuto (Q8) e 2.000 com mais de 1 dia, incluindo um valor de 2.147.483.647 minutos, que é erro de cadastro. A média é 9.398 minutos e a mediana, 40 (Q9).
- **Outros domínios:** 1 receita sem passos (Q10). Datas entre 06/08/1999 e 04/12/2018 (Q12). `n_steps` e `n_ingredients` batem com o tamanho das listas (Q13). Todas as receitas têm pelo menos 1 tag (Q15).
- **Calorias:** máximo de 434.360 kcal contra mediana de 313 kcal (Q14).

## 4. Limpeza e preparação dos dados

Realize a limpeza necessária ao projeto. Verifique valores ausentes, duplicados e inconsistências e trate os problemas encontrados, quando necessário.

In [ ]:
df = df_original.copy()
print("Linhas no início:", len(df))

Linhas no início: 231637


In [ ]:
# D1 remover receitas duplicadas
antes = len(df)
df = df.drop_duplicates(subset=["name", "contributor_id"], keep="first")
print("Linhas removidas:", antes - len(df))
print("Duplicadas restantes:", df.duplicated(subset=["name", "contributor_id"]).sum())

Linhas removidas: 5
Duplicadas restantes: 0


In [ ]:
# D2 remover receita sem nome e receita sem passos
antes = len(df)
df = df[df["name"].notna() & (df["n_steps"] > 0)]
print("Linhas removidas:", antes - len(df))
print("Sem nome restantes:", df["name"].isna().sum())
print("Sem passos restantes:", (df["n_steps"] == 0).sum())

Linhas removidas: 2
Sem nome restantes: 0
Sem passos restantes: 0


In [ ]:
# D3 description mantida
print("Receitas sem description (mantidas):", df["description"].isna().sum())

Receitas sem description (mantidas): 4979


In [ ]:
# D4 trocar tempos impossíveis por vazio (NaN)
df.loc[df["minutes"] == 0, "minutes"] = np.nan
df.loc[df["minutes"] == 2147483647, "minutes"] = np.nan
print("Tempos agora vazios:", df["minutes"].isna().sum())
print("Menor tempo:", df["minutes"].min())
print("Maior tempo:", df["minutes"].max())
print("Média:", round(df["minutes"].mean(), 2), "| Mediana:", df["minutes"].median())

Tempos agora vazios: 1095
Menor tempo: 1.0
Maior tempo: 1051200.0
Média: 128.25 | Mediana: 40.0


**Resposta do grupo:**

A limpeza foi feita numa cópia (`df`), mantendo `df_original` intacto para comparação.

- **D1 (Q7):** 5 receitas duplicadas removidas, mantendo a primeira cadastrada.
- **D2 (Q1 e Q10):** removidas a receita sem nome e a receita sem passos (2 registros).
- **D3 (Q2):** receitas sem descrição mantidas. A descrição é opcional, e remover esses registros descartaria 4.979 receitas válidas.
- **D4 (Q8 e Q9):** os 1.094 tempos iguais a 0 e o valor de 2.147.483.647 foram substituídos por ausente (1.095 no total), sem remover as receitas. Os demais tempos longos foram mantidos, porque parte deles é real (fermentações, licores). Nas análises de tempo, usar a mediana.
- **D5 (Q14):** calorias extremas mantidas. O valor por porção depende do rendimento informado pelo autor, e não há como separar erro de valor legítimo.

**Resultado:** a base passou de 231.637 para 231.630 receitas (7 removidas, 0,003% de 231.637).

## 5. Transformação dos dados

Realize as transformações necessárias à análise, como criação de colunas, filtros, agrupamentos, conversões ou junções, quando aplicáveis.

In [ ]:
# D6 converter tipos e formatos
df["submitted"] = pd.to_datetime(df["submitted"])

for coluna in ["tags", "steps", "ingredients", "nutrition"]:
    df[coluna] = df[coluna].apply(ast.literal_eval)

colunas_nutri = ["calories", "total_fat_pdv", "sugar_pdv", "sodium_pdv", "protein_pdv", "saturated_fat_pdv", "carbohydrates_pdv"]
df[colunas_nutri] = pd.DataFrame(df["nutrition"].tolist(), index=df.index)

print("Tipo de submitted:", df["submitted"].dtype)
print("Tipo de tags:", type(df["tags"].iloc[0]))
display(df[colunas_nutri].head())

Tipo de submitted: datetime64[ns]
Tipo de tags: <class 'list'>


,calories,total_fat_pdv,sugar_pdv,sodium_pdv,protein_pdv,saturated_fat_pdv,carbohydrates_pdv
0,51.5,0.0,13.0,0.0,2.0,0.0,4.0
1,173.4,18.0,0.0,17.0,22.0,35.0,1.0
2,269.8,22.0,32.0,48.0,39.0,27.0,5.0
3,368.1,17.0,10.0,2.0,14.0,8.0,20.0
4,352.9,1.0,337.0,23.0,3.0,0.0,28.0


In [ ]:
# Salva a base
os.makedirs("data", exist_ok=True)
CAMINHO_FINAL = "data/recipes_ra1.parquet"
df.to_parquet(CAMINHO_FINAL, index=False)


df_final = pd.read_parquet(CAMINHO_FINAL)
assert df_final.shape == (231630, 19), "A base final não é a esperada (231.630 × 19)"
print(f"✓ Arquivo salvo: {CAMINHO_FINAL}")
print(f"Shape: {df_final.shape[0]} linhas × {df_final.shape[1]} colunas")
df_final.info()

✓ Arquivo salvo: data/recipes_ra1.parquet
Shape: 231630 linhas × 19 colunas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231630 entries, 0 to 231629
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   name               231630 non-null  object        
 1   id                 231630 non-null  int64         
 2   minutes            230535 non-null  float64       
 3   contributor_id     231630 non-null  int64         
 4   submitted          231630 non-null  datetime64[ns]
 5   tags               231630 non-null  object        
 6   nutrition          231630 non-null  object        
 7   n_steps            231630 non-null  int64         
 8   steps              231630 non-null  object        
 9   description        226651 non-null  object        
 10  ingredients        231630 non-null  object        
 11  n_ingredients      231630 non-null  int64         
 12  calories           23163

In [ ]:
# Verificações finais
print("--- Duplicidades ---")
print("ids repetidos:", df_final["id"].duplicated().sum())
print("nome + autor repetidos:", df_final.duplicated(subset=["name", "contributor_id"]).sum())

print("\n--- Ausências ---")
display(df_final.isna().sum())

print("\n--- Domínios ---")
print("minutes igual a 0:", (df_final["minutes"] == 0).sum())
print("minutes máximo:", df_final["minutes"].max())
print("n_steps igual a 0:", (df_final["n_steps"] == 0).sum())
print("n_ingredients igual a 0:", (df_final["n_ingredients"] == 0).sum())
print("Período:", df_final["submitted"].min(), "a", df_final["submitted"].max())
print("Valores negativos por coluna numérica:")
display(df_final.select_dtypes("number").lt(0).sum())

print("\n--- Integridade ---")
print("n_steps diferente da qtd de passos:", (df_final["n_steps"] != df_final["steps"].apply(len)).sum())
print("n_ingredients diferente da qtd de ingredientes:", (df_final["n_ingredients"] != df_final["ingredients"].apply(len)).sum())

--- Duplicidades ---
ids repetidos: 0
nome + autor repetidos: 0

--- Ausências ---


,0
name,0
id,0
minutes,1095
contributor_id,0
submitted,0
tags,0
nutrition,0
n_steps,0
steps,0
description,4979



--- Domínios ---
minutes igual a 0: 0
minutes máximo: 1051200.0
n_steps igual a 0: 0
n_ingredients igual a 0: 0
Período: 1999-08-06 00:00:00 a 2018-12-04 00:00:00
Valores negativos por coluna numérica:


,0
id,0
minutes,0
contributor_id,0
n_steps,0
n_ingredients,0
calories,0
total_fat_pdv,0
sugar_pdv,0
sodium_pdv,0
protein_pdv,0



--- Integridade ---
n_steps diferente da qtd de passos: 0
n_ingredients diferente da qtd de ingredientes: 0


**Resposta do grupo:**

- **D6 (Q4 e Q5):** `submitted` foi convertida para data, e `tags`, `steps`, `ingredients` e `nutrition` foram convertidas para listas. A `nutrition` foi separada em 7 colunas numéricas: `calories` (kcal) e `total_fat_pdv`, `sugar_pdv`, `sodium_pdv`, `protein_pdv`, `saturated_fat_pdv` e `carbohydrates_pdv` (% do valor diário). A coluna original foi mantida para rastreabilidade. Sem essa conversão não é possível analisar por período, contar tags e ingredientes nem usar os valores nutricionais.
- **Saída:** a base final, com 231.630 linhas e 19 colunas, foi salva em `data/recipes_ra1.parquet`. O Parquet preserva os tipos (data e listas), o que o CSV não faz.
- **Verificações finais:** 0 duplicatas, 0 valores zerados ou negativos e 0 divergências entre as contagens e as listas. As únicas ausências são as esperadas: 1.095 em `minutes` (D4) e 4.979 em `description` (D3).


## 7. Considerações do RA1

Registre brevemente:

- O que o grupo descobriu sobre os dados.
- Quais dificuldades ou limitações encontrou.
- Como os dados preparados poderão ser utilizados nas próximas etapas.

**Resposta do grupo:**

**O que o grupo descobriu:** na maior parte, a base é consistente: o `id` é único, as datas estão dentro do período, as contagens de passos e ingredientes batem com as listas e toda receita tem pelo menos uma tag. Os problemas se concentraram em `minutes`, nas colunas guardadas como texto e em poucos registros duplicados ou incompletos. Apenas 7 receitas precisaram ser removidas.

**Dificuldades e limitações:**
- `minutes` ainda tem valores muito altos que podem ser reais, então análises de tempo devem usar a mediana. As 1.095 receitas sem tempo ficam fora dessas análises.
- As calorias extremas foram mantidas, porque o valor por porção depende do rendimento informado pelo autor.
- `calories` está em kcal e as outras colunas nutricionais em % do valor diário, então não podem ser somadas nem comparadas diretamente.
- A culinária e as restrições alimentares ainda estão dentro de `tags`, sem coluna própria.

**Uso nas próximas etapas:** a base em Parquet mantém os tipos e é compatível com o Spark. Com `submitted` em formato de data, é possível simular a chegada das receitas em ordem cronológica no processamento em streaming do RA2 e aplicar janelas de tempo. As colunas de tempo, complexidade e nutrição, junto com a extração da culinária, permitem responder às perguntas P1 a P8.

---

# RA2 — Streaming Data Processing and Time Windows

Utilize os dados e o contexto do projeto para demonstrar o processamento em streaming e a aplicação de janelas temporais. Escolha um dos cenários:

- **Cenário A — Dados estáticos:** simule a chegada de novos registros utilizando dados artificiais coerentes com o contexto do projeto.
- **Cenário B — Dados em streaming:** utilize sua própria fonte de dados contínuos, sem necessidade de simulação.

## 8. Objetivo da análise em streaming

Responda brevemente:

- O que o grupo deseja monitorar ou analisar?
- Qual informação espera obter com o processamento em streaming?

**Resposta do grupo:**

[Preencha aqui.]

## 9. Fonte de dados e processamento

- Informe o cenário escolhido: dados reais em streaming ou simulação.
- Explique brevemente como os novos registros chegam.
- Implemente o processamento em tempo real.
- Demonstre o recebimento de novos registros.

**Resposta do grupo:**

[Preencha aqui.]

In [ ]:
# Utilize quantas células de código forem necessárias para esta etapa.

## 10. Aplicação de janelas temporais

- Utilize uma janela **Tumbling** (intervalos sem sobreposição) ou **Sliding** (intervalos que podem se sobrepor).
- Informe o tipo e a duração da janela e o deslocamento, quando aplicável.
- Implemente uma agregação coerente com a análise, como contagem, soma ou média.
- Explique brevemente o que será calculado em cada intervalo.

O foco é demonstrar a compreensão do conceito; não é necessário justificar tecnicamente a duração escolhida.

**Resposta do grupo:**

[Informe o tipo, a duração, o deslocamento (quando aplicável) e o que será calculado em cada janela.]

In [ ]:
# Utilize quantas células de código forem necessárias para esta etapa.

## 11. Resultados e interpretação

Apresente os resultados do processamento e demonstre como a análise se comporta com a chegada de novos registros.

In [ ]:
# Utilize quantas células de código forem necessárias para esta etapa.

**Resposta do grupo:**

[Explique brevemente o significado dos resultados e as mudanças observadas com a chegada de novos registros.]

## 12. Considerações do RA2

Responda brevemente:

- O que foi possível observar com o processamento em streaming?
- Qual foi a contribuição das janelas temporais para a análise?
- Essa abordagem faria sentido em uma aplicação real do projeto? Por quê?

Na última resposta, o grupo pode reconhecer que o streaming foi utilizado apenas como exercício didático, se esse for o caso.

**Resposta do grupo:**

[Preencha aqui.]